In [29]:
import pandas as pd
import numpy as np
import duckdb
import logging

In [30]:
# Load Fire and Resource Assessment Program (FRAP) dataset
frap = pd.read_csv('https://gis.data.cnra.ca.gov/api/download/v1/items/c3c10388e3b24cec8a954ba10458039d/csv?layers=0')

# Select relevant columns for fire table
fires_df = frap.drop(columns=['Complex ID', 'DECADES', 'Comments', 'Fire Number (historical use)', 'Complex Name', 
                              'Management Objective', 'Local Incident Number', 'IRWIN ID', 'Year', 
                              'Fire Name', 'Unit ID'])

# Rename columns for consistency
fires_df.rename(columns={
    'OBJECTID': 'fire_id',
    'State': 'state',
    'Agency': 'agency_name',
    'Alarm Date': 'alarm_date',
    'Containment Date': 'containment_date',
    'Cause': 'cause_id',
    'Collection Method': 'collection_method',
    'GIS Calculated Acres': 'acres_burned',
    'Shape__Area': 'shape_area',
    'Shape__Length': 'shape_length'
}, inplace=True)

# Convert dates to datetime
fires_df['alarm_date'] = pd.to_datetime(fires_df['alarm_date'], errors='coerce')
fires_df['containment_date'] = pd.to_datetime(fires_df['containment_date'], errors='coerce')

# Compute duration_days
fires_df['duration_days'] = (fires_df['containment_date'] - fires_df['alarm_date']).dt.days

fires_df.head()

,fire_id,state,agency_name,alarm_date,containment_date,cause_id,collection_method,acres_burned,shape_area,shape_length,duration_days
0,1,CA,CDF,2025-01-07 08:00:00,2025-01-31 08:00:00,14,7.0,23448.8800,1.386518e+08,140231.608232,24.0
1,2,CA,CDF,2025-01-08 08:00:00,2025-01-31 08:00:00,14,7.0,14056.2600,8.336393e+07,104933.207224,23.0
2,3,CA,CDF,2025-01-22 08:00:00,2025-01-28 08:00:00,14,7.0,10396.8000,6.216064e+07,96698.599858,6.0
3,4,CA,CCO,2025-01-09 08:00:00,2025-02-04 08:00:00,14,2.0,998.7378,5.919678e+06,15602.004849,26.0
4,5,CA,CDF,2025-01-07 08:00:00,2025-01-09 08:00:00,14,7.0,831.3855,4.946082e+06,16094.217073,2.0


In [31]:
# Create Causes and Agency tables

# Create Causes table
causes_df = fires_df[['cause_id']].drop_duplicates().reset_index(drop=True)

# Mapping for descriptive names
cause_mapping = {1: "Lightning",
                 2: "Equipment Use",
                 3: "Smoking",
                 4: "Campfire",
                 5: "Debris",
                 6: "Railroad",
                 7: "Arson",
                 8: "Playing with fire",
                 9: "Miscellaneous",
                 10: "Vehicle",
                 11: "Powerline",
                 12: "Firefighter Training",
                 13: "Non-Firefighter Training",
                 14: "Unknown/Unidentified",
                 15: "Structure",
                 16: "Aircraft",
                 17: "Volcanic",
                 18: "Escaped Prescribed Burn",
                 19: "Illegal Alien Campfire"}

# Map cause_id to cause_name
causes_df['cause_name'] = causes_df['cause_id'].map(cause_mapping)

# Make cause_type column with natural or human cause types
def classify_cause(cause_id):
    if cause_id in [1, 17]:  # Natural causes
        return 'Natural'
    elif cause_id in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19]:  # Human causes
        return 'Human'
    else:
        return 'Unknown'
    
causes_df['cause_type'] = causes_df['cause_id'].apply(classify_cause)

# Sort causes by cause_id
causes_df.sort_values('cause_id', inplace=True)
causes_df.head()

,cause_id,cause_name,cause_type
2,1,Lightning,Natural
3,2,Equipment Use,Human
12,3,Smoking,Human
10,4,Campfire,Human
9,5,Debris,Human


In [32]:
# Agency table
agency_df = fires_df[['agency_name']].drop_duplicates().reset_index(drop=True)
agency_df['agency_id'] = agency_df.index + 1  # simple PK
agency_df = agency_df[['agency_id', 'agency_name']]

# Mapping display names to stored codes, based off data dictionary 
# https://34c031f8-c9fd-4018-8c5a-4159cdff6b0d-cdn-endpoint.azureedge.net/-/media/calfire-website/what-we-do/fire-resource-assessment-program---frap/data-dictionary-updated-april-2025.pdf?rev=99e4dccd9d654a6f9ed9556c0b7e3445&hash=371C2A47D6A96261C91E0008A1BABC20
agency_code_mapping = {
    "BIA": "USDI Bureau of Indian Affairs",
    "BLM": "Bureau of Land Management",
    "CDF": "California Department of Forestry and Fire Protection",
    "CCO": "Contract County",
    "CSP": "California State Parks",
    "DOD": "Department of Defense",
    "FWS": "USDI Fish and Wildlife Service",
    "LRA": "Local Responsibility Area",
    "NOP": "No Protection",
    "NPS": "National Park Service",
    "PVT": "Private",
    "USF": "USDA Forest Service",
    "OTH": "Other"}

# Add the stored code column
agency_df['agency_name_long'] = agency_df['agency_name'].map(agency_code_mapping)

# Optional: reorder columns
agency_df = agency_df[['agency_id', 'agency_name', 'agency_name_long']]

# Display the table
agency_df.head(15)

## idea, maybe find agency informations from the web and add more columns to the agency table, such as agency type (federal, state, local), funding, number of personnel, etc. This would allow for more in-depth analysis of how different types of agencies respond to fires and their effectiveness.

,agency_id,agency_name,agency_name_long
0,1,CDF,California Department of Forestry and Fire Pro...
1,2,CCO,Contract County
2,3,USF,USDA Forest Service
3,4,NPS,National Park Service
4,5,BLM,Bureau of Land Management
5,6,LRA,Local Responsibility Area
6,7,BIA,USDI Bureau of Indian Affairs
7,8,DOD,Department of Defense
8,9,FWS,USDI Fish and Wildlife Service
9,10,NaN,NaN


In [33]:
# Time table
# Extract unique alarm dates
time_df = fires_df[['alarm_date']].drop_duplicates().reset_index(drop=True)

# Extract year, month, decade 
time_df['year'] = time_df['alarm_date'].dt.year
time_df['month'] = time_df['alarm_date'].dt.month
time_df['decade'] = (time_df['year'] // 10) * 10
time_df['day_of_year'] = time_df['alarm_date'].dt.dayofyear
time_df['week'] = time_df['alarm_date'].dt.isocalendar().week
time_df['is_peak_fire_season'] = time_df['month'].isin([6,7,8,9]).astype(int)

# Define seasons using a function
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:  # 9, 10, 11
        return 'Fall'

time_df['season'] = time_df['month'].apply(get_season)

# Add a primary key for relational mapping
time_df['time_id'] = time_df.index + 1

# Reorder columns
time_df = time_df[['time_id', 'alarm_date', 'year', 'month', 'decade', 'day_of_year', 'week', 'season', 'is_peak_fire_season']]

In [34]:
fires_df = fires_df.merge(causes_df, on='cause_id', how='left')
fires_df = fires_df.merge(agency_df, on='agency_name', how='left')
fires_df = fires_df.merge(time_df[['alarm_date', 'time_id']], on='alarm_date', how='left')
fires_df = fires_df[fires_df['state'] == 'CA'].reset_index(drop=True)

In [35]:
fires_df = fires_df.drop(columns=[
    'agency_name',
    'agency_name_long',
    'cause_name',
    'cause_type',
    'alarm_date',
    'containment_date',
    'state'])

fires_df = fires_df[['fire_id', 'cause_id', 'agency_id', 'time_id', 'collection_method', 'acres_burned', 'shape_area', 'shape_length', 'duration_days']]

In [36]:
fires_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22794 entries, 0 to 22793
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   fire_id            22794 non-null  int64  
 1   cause_id           22794 non-null  int64  
 2   agency_id          22794 non-null  int64  
 3   time_id            22794 non-null  int64  
 4   collection_method  10695 non-null  float64
 5   acres_burned       22794 non-null  float64
 6   shape_area         22794 non-null  float64
 7   shape_length       22794 non-null  float64
 8   duration_days      10159 non-null  float64
dtypes: float64(5), int64(4)
memory usage: 1.6 MB


In [37]:
# Connect and create tables
# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def create_database():
    try:
        # Connect to local DuckDB instance
        con = duckdb.connect(database='california_fires.duckdb', read_only=False)
        logging.info("Connected to DuckDB database 'california_fires.duckdb'")
        print("Connected to DuckDB database 'california_fires.duckdb'")

        # Drop tables if they already exist
        con.execute("DROP TABLE IF EXISTS Fires")
        con.execute("DROP TABLE IF EXISTS Causes")
        con.execute("DROP TABLE IF EXISTS Agency")
        con.execute("DROP TABLE IF EXISTS Time")
        logging.info("Existing tables dropped if they existed.")

        # Create tables
        con.execute("CREATE TABLE Causes AS SELECT * FROM causes_df")
        con.execute("CREATE TABLE Agency AS SELECT * FROM agency_df")
        con.execute("CREATE TABLE Time AS SELECT * FROM time_df")
        con.execute("CREATE TABLE Fires AS SELECT * FROM fires_df")
        logging.info("Tables created successfully in DuckDB database.")
        print("Tables created successfully in DuckDB database.")

    except Exception as e:
        logging.error(f"An error occurred: {e}")
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    create_database()

2026-03-22 15:36:35,494 - INFO - Connected to DuckDB database 'california_fires.duckdb'
2026-03-22 15:36:35,501 - INFO - Existing tables dropped if they existed.
2026-03-22 15:36:35,522 - INFO - Tables created successfully in DuckDB database.


Connected to DuckDB database 'california_fires.duckdb'
Tables created successfully in DuckDB database.


In [40]:
# Save each table as a CSV file
fires_df.to_csv('./CSV_files/fires.csv', index=False)
causes_df.to_csv('./CSV_files/causes.csv', index=False)
agency_df.to_csv('./CSV_files/agency.csv', index=False)
time_df.to_csv('./CSV_files/time.csv', index=False)
logging.info("Tables saved as CSV files.")
print("Tables saved as CSV files.")

2026-03-22 15:41:22,094 - INFO - Tables saved as CSV files.


Tables saved as CSV files.


In [38]:
# # Save each table as parquet file
# fires_df.to_parquet('fires.parquet', index=False)
# causes_df.to_parquet('causes.parquet', index=False)
# agency_df.to_parquet('agency.parquet', index=False)
# time_df.to_parquet('time.parquet', index=False)

# logging.info("All DataFrames saved as parquet files.")
# print("All DataFrames saved as parquet files.")

2026-03-22 15:36:35,830 - INFO - All DataFrames saved as parquet files.


All DataFrames saved as parquet files.
